# 21 — Cohort-rule sensitivity (audit FIX-01)

The 2026-08-01 integrity audit found that notebook 03's `fl_client` rule reads the sealed year:
eligibility requires reporting in the panel's **final month (March 2026)** and counts history over
all 96 months — information unknowable at the development cut. This is selection look-ahead
(no future *values* enter any model, but the cohort itself was chosen with future knowledge).

This notebook is the audit-directed sensitivity, not a repair-in-place:

- `src/17_sealed_year_eval.ipynb` and every stored csv **stand verbatim as the pre-registered
  record**; nothing upstream is re-run or overwritten.
- The strictly prospective rule (same thresholds, history measured before `TEST_START` only) is
  derived here, the moved providers are named, and the anchor numbers are recomputed under both
  rules from one shared implementation (fitting code verbatim from notebook 11).
- **The last cell reads FY 2025-26 rows** — the second reading of the sealed year, authorized by
  the audit fix plan and recorded in `docs/WORKLOG.md`. It scores only deterministic persistence.

Output: `data/processed/cohort_sensitivity.csv` — the both-rules table the paper's disclosure
cites. Headline: validation anchor 153.49 (n=1,407) → 154.71 (n=1,418) without the look-ahead;
sealed anchor 151.50 (n=1,407) → 151.34 (n=1,409); model ordering unchanged, so every structural
conclusion survives; the sealed "persistence by 0.1 MAE" verdict is inside cohort sensitivity.

In [1]:
# Goal: quantify the audit's H1 finding. The stored fl_client rule (notebook 03) requires
# reporting in the panel's final month (March 2026, inside the sealed year) and counts history
# over all 96 months - so cohort membership used information unknowable at the development cut.
# This cell derives the strictly prospective rule (history measured before TEST_START only) and
# measures exactly which providers the two rules disagree on.

from pathlib import Path   # stdlib paths: keeps the ../data locations readable
import numpy as np         # least-squares fits and error arithmetic in the next cells
import pandas as pd        # dataframes: the canonical panel and every comparison table

PROC = Path("../data/processed")
TEST_START = pd.Timestamp("2025-04-01")      # the declared sealed cut, same constant as 04-19
VAL_START = pd.Timestamp("2024-04-01")       # validation year FY 2024-25, unchanged

panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
assert panel.shape == (343441, 24), panel.shape   # a regenerated panel csv trips this

# the stored rule, as shipped by notebook 03: flags already sit on the panel
stored = panel[panel["fl_client"]].groupby("provider_code")["is_public"].first()
assert len(stored) == 392 and int(stored.sum()) == 149   # the documented stored pool

# the prospective rule: same thresholds, history measured inside the development window only
dev = panel[panel["month"] < TEST_START]     # Apr 2018 - Mar 2025: all a modeller could know
ds = dev.groupby("provider_code")["month"].agg(n_dev="nunique", last_dev="max")
pub = panel.groupby("provider_code")["is_public"].first()
devcut = ds[(ds["n_dev"] >= 48) & (ds["last_dev"] == pd.Timestamp("2025-03-01"))].index
assert len(devcut) == 377 and int(pub.loc[devcut].sum()) == 150   # the audit's quantification,
# re-derived here from the panel; a changed panel or rule trips this

kept_by_sealed = sorted(set(stored.index) - set(devcut))   # in the study only because of FY 2025-26
dropped_retro = sorted(set(devcut) - set(stored.index))    # prospective members dropped retroactively
assert len(kept_by_sealed) == 22 and len(dropped_retro) == 7
assert "RW4" in kept_by_sealed    # Mersey Care: 42 dev months, crossed 48 during the sealed year
assert "RHQ" in dropped_retro     # Sheffield Teaching Hospitals: 84 dev months, stopped
assert "RY6" in dropped_retro     # reporting during FY 2025-26 - as did Leeds Community
assert ds.loc["RW4", "n_dev"] == 42 and ds.loc["RHQ", "n_dev"] == 84

print(f"stored rule: {len(stored)} clients ({int(stored.sum())} public); "
      f"prospective rule: {len(devcut)} ({int(pub.loc[devcut].sum())} public)")
print(f"kept only via sealed-year months: {len(kept_by_sealed)} (incl RW4); "
      f"dropped retroactively: {len(dropped_retro)} (incl RHQ, RY6)")

stored rule: 392 clients (149 public); prospective rule: 377 (150 public)
kept only via sealed-year months: 22 (incl RW4); dropped retroactively: 7 (incl RHQ, RY6)


In [2]:
# Goal: the validation anchor and the full linear family under both cohort rules, from one
# implementation (fitting code verbatim from notebook 11, full-history regime). Scoring the
# stored cohort first and asserting it reproduces fl_summary_c110.csv proves this replication
# is faithful; only then is the prospective cohort's column trustworthy.

def anchor_rows(cohort):
    # notebook 11's row construction under an explicit provider cohort: features and targets
    # built inside the development window only, validation = FY 2024-25 origins
    df = dev[(dev["tfunc_code"] == "C_110") & dev["provider_code"].isin(cohort)
             & dev["is_public"]].sort_values(["provider_code", "month"]).copy()
    g = df.groupby("provider_code")["total_waiting"]
    df["target"] = g.shift(-1)                       # next month's list
    df["delta_1"] = df["total_waiting"] - g.shift(1) # 1-month momentum
    df = df.dropna(subset=["target"])
    tr = df[df["month"] < VAL_START].dropna(subset=["delta_1"])
    val = df[df["month"] >= VAL_START].copy()
    # RJ8's first C_110 return is May 2024, so its validation momentum is unknown: zero change
    # is the neutral reading (same treatment as 11)
    val["delta_1"] = np.where(val["delta_1"].notna(), val["delta_1"], 0.0)
    return tr, val


FEATS = ["total_waiting", "delta_1"]

def fit(rows):                               # least squares with an intercept column of 1s
    X = np.column_stack([np.ones(len(rows)), rows[FEATS].to_numpy()])
    return np.linalg.lstsq(X, rows["target"].to_numpy(), rcond=None)[0]

def fit_delta(rows):                         # change = a + b * last month's change
    X = np.column_stack([np.ones(len(rows)), rows["delta_1"].to_numpy()])
    return np.linalg.lstsq(X, (rows["target"] - rows["total_waiting"]).to_numpy(), rcond=None)[0]

def family_mae(cohort):
    # every model of notebook 11, full-history regime, scored on the cohort's validation rows
    tr, val = anchor_rows(cohort)
    truth = np.concatenate([r["target"].to_numpy() for _, r in val.groupby("provider_code")])
    cur = np.concatenate([r["total_waiting"].to_numpy() for _, r in val.groupby("provider_code")])
    betas = {p: (fit(r), len(r)) for p, r in tr.groupby("provider_code")}
    bd = {p: (fit_delta(r), len(r)) for p, r in tr.groupby("provider_code")}
    bfl = sum(b * n for b, n in betas.values()) / sum(n for _, n in betas.values())
    bfd = sum(b * n for b, n in bd.values()) / sum(n for _, n in bd.values())
    X = lambda r: np.column_stack([np.ones(len(r)), r[FEATS].to_numpy()])
    preds = {
        "persistence": cur,
        "pooled": np.concatenate([X(r) @ fit(tr) for _, r in val.groupby("provider_code")]),
        "fedavg-delta": np.concatenate(
            [r["total_waiting"].to_numpy() + bfd[0] + bfd[1] * r["delta_1"].to_numpy()
             for _, r in val.groupby("provider_code")]),
        "fedavg-delta+intercept": np.concatenate(
            [r["total_waiting"].to_numpy() + (bd[p][0][0] if p in bd else 0.0)   # RJ8: no local
             + bfd[1] * r["delta_1"].to_numpy()                                  # intercept yet
             for p, r in val.groupby("provider_code")]),
        "local-only": np.concatenate(
            [X(r) @ betas[p][0] if p in betas else r["total_waiting"].to_numpy()  # RJ8 falls
             for p, r in val.groupby("provider_code")]),                          # back to naive
        "fedavg-level": np.concatenate([X(r) @ bfl for _, r in val.groupby("provider_code")]),
    }
    return len(val), val["provider_code"].nunique(), {m: np.abs(truth - p).mean() for m, p in preds.items()}


n_st, p_st, mae_st = family_mae(stored.index)
n_dc, p_dc, mae_dc = family_mae(devcut)

# the stored-cohort replication must equal the stored csv before the dev-cut column means anything
flsum = pd.read_csv(PROC / "fl_summary_c110.csv").set_index(["regime", "model"])["mae"]
CSV_KEYS = {"persistence": ("-", "naive persistence"), "pooled": ("full", "pooled"),
            "fedavg-delta": ("full", "fedavg-delta"),
            "fedavg-delta+intercept": ("full", "fedavg-delta+local-a"),
            "local-only": ("full", "local-only"), "fedavg-level": ("full", "fedavg")}
assert n_st == 1407
for m, key in CSV_KEYS.items():
    assert abs(mae_st[m] - flsum[key]) < 0.01, (m, mae_st[m], flsum[key])   # replication drift trips

assert n_dc == 1418 and p_dc == 129          # the prospective cohort adds RHQ's 11 rows
assert round(mae_dc["persistence"], 1) == 154.7   # the anchor without the look-ahead flattering
order = ["persistence", "pooled", "fedavg-delta", "fedavg-delta+intercept", "local-only", "fedavg-level"]
assert all(mae_dc[a] < mae_dc[b] for a, b in zip(order, order[1:]))   # the paper's model ordering
# must survive the cohort fix - it does; a reordering under the prospective rule would trip this

print(f"{'model':24s} {'stored (n=1,407)':>17s} {'prospective (n=1,418)':>22s}")
for m in order:
    print(f"{m:24s} {mae_st[m]:17.2f} {mae_dc[m]:22.2f}")

model                     stored (n=1,407)  prospective (n=1,418)
persistence                         153.49                 154.71
pooled                              153.76                 155.04
fedavg-delta                        153.98                 155.29
fedavg-delta+intercept              155.28                 156.56
local-only                          161.86                 163.20
fedavg-level                        186.45                 187.58


In [3]:
# Goal: the sealed-anchor sensitivity and the stored output. THIS CELL READS FY 2025-26 ROWS -
# the second reading of the sealed year, after 17's one-shot. It is not a re-run of the one-shot
# (17's csv stands verbatim as the pre-registered record) but the audit-directed sensitivity:
# the same persistence score under the cohort a modeller could actually have declared.

def sealed_persistence(cohort):
    # 17's persistence construction: origins in FY 2025-26, target = next month, per provider
    df = panel[(panel["tfunc_code"] == "C_110") & panel["provider_code"].isin(cohort)
               & panel["is_public"]].sort_values(["provider_code", "month"]).copy()
    df["target"] = df.groupby("provider_code")["total_waiting"].shift(-1)
    s = df[(df["month"] >= TEST_START) & df["target"].notna()]
    return len(s), np.abs(s["target"] - s["total_waiting"]).mean()


sn_st, smae_st = sealed_persistence(stored.index)
sn_dc, smae_dc = sealed_persistence(devcut)

sealed_csv = pd.read_csv(PROC / "sealed_year_eval.csv")
csv_pers = sealed_csv[(sealed_csv["tfunc_code"] == "C_110")
                      & (sealed_csv["model"] == "persistence")]["mae"].iloc[0]
assert sn_st == 1407 and abs(smae_st - csv_pers) < 0.01   # must reproduce 17's stored one-shot
assert sn_dc == 1409 and round(smae_dc, 1) == 151.3       # the prospective-cohort sensitivity

# the paper's sealed verdict was persistence over neural pooled by 0.11 MAE; the cohort rule
# alone moves persistence by 0.16 - the verdict's sign is inside cohort sensitivity
neural_csv = sealed_csv[(sealed_csv["tfunc_code"] == "C_110")
                        & (sealed_csv["model"] == "neural_pooled")]["mae"].iloc[0]
assert abs(smae_st - smae_dc) > abs(neural_csv - smae_st)

rows = [
    ("fl_client providers (all sectors)", len(stored), len(devcut)),
    ("fl_client providers (public)", int(stored.sum()), int(pub.loc[devcut].sum())),
    ("validation rows (C_110)", n_st, n_dc),
    ("validation providers (C_110)", p_st, p_dc),
    ("validation persistence MAE", round(mae_st["persistence"], 2), round(mae_dc["persistence"], 2)),
    ("validation pooled MAE (full)", round(mae_st["pooled"], 2), round(mae_dc["pooled"], 2)),
    ("validation fedavg-delta MAE (full)", round(mae_st["fedavg-delta"], 2), round(mae_dc["fedavg-delta"], 2)),
    ("validation fedavg-delta+intercept MAE (full)",
     round(mae_st["fedavg-delta+intercept"], 2), round(mae_dc["fedavg-delta+intercept"], 2)),
    ("validation local-only MAE (full)", round(mae_st["local-only"], 2), round(mae_dc["local-only"], 2)),
    ("validation fedavg-level MAE (full)", round(mae_st["fedavg-level"], 2), round(mae_dc["fedavg-level"], 2)),
    ("sealed rows (C_110)", sn_st, sn_dc),
    ("sealed persistence MAE", round(smae_st, 2), round(smae_dc, 2)),
]
out = pd.DataFrame(rows, columns=["quantity", "stored_rule", "prospective_rule"])
out.to_csv(PROC / "cohort_sensitivity.csv", index=False)
print(out.to_string(index=False))
print(f"\nverdict: the sealed anchor margin (persistence {smae_st:.2f} vs neural pooled "
      f"{neural_csv:.2f}, gap {neural_csv - smae_st:+.2f}) is smaller than the cohort-rule "
      f"effect on persistence alone ({smae_dc - smae_st:+.2f}) - the anchor verdict is a tie, "
      "not a win. Model ordering and all structural conclusions are unchanged (cell above).")
print("->", PROC / "cohort_sensitivity.csv", out.shape)

                                    quantity  stored_rule  prospective_rule
           fl_client providers (all sectors)       392.00            377.00
                fl_client providers (public)       149.00            150.00
                     validation rows (C_110)      1407.00           1418.00
                validation providers (C_110)       128.00            129.00
                  validation persistence MAE       153.49            154.71
                validation pooled MAE (full)       153.76            155.04
          validation fedavg-delta MAE (full)       153.98            155.29
validation fedavg-delta+intercept MAE (full)       155.28            156.56
            validation local-only MAE (full)       161.86            163.20
          validation fedavg-level MAE (full)       186.45            187.58
                         sealed rows (C_110)      1407.00           1409.00
                      sealed persistence MAE       151.50            151.34

verdict: th